# 13 · Stage 3 v5-A — spatial V-JEPA + T=32 + A2D2 mixed training

목표는 v4-A의 단순 calibration 미세조정이 아니라 표현/데이터 병목을 직접 건드리는 것이다.

핵심 변경:
- V-JEPA 2.1 spatial token 전체 평균을 없애고 3×4 spatial moments를 학습 가능한 residual로 사용
- clip length 16 → 32
- comma2k19 + A2D2 mixed training
- rare event / A2D2 event-balanced sampling
- STOP multi-threshold ordinal
- explicit Δspeed head
- yaw-turn ordinal + steering direction/activity auxiliary
- A2D2 brake/throttle auxiliary
- v4-A CAN head warm-start, V-JEPA backbone은 우선 frozen

`stage3/metrics.py`는 수정하지 않는다. 아래 threshold들은 학습용 물리적 auxiliary이며 DACON hidden threshold라고 가정하지 않는다.


In [2]:
from __future__ import annotations

import copy
import json
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

# Keep Colab binary stack; install only small pure/python extras.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "wandb==0.29.0",
        "easydict==1.13",
    ],
    check=True,
)

# Do not depend on editable-install visibility in Colab 25.10.
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"

COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
A2D2_ROOT = DATA_ROOT / "A2D2" / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests" / "stage3" / "v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5a.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

# Runtime normalization injection for the inherited v4 loss.
loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["base"]["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

print("Python         :", sys.version)
print("Torch          :", torch.__version__)
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Git            :", BRANCH, GIT_COMMIT)
print("Config         :", CFG_PATH)
print("comma root     :", COMMA_ROOT)
print("A2D2 root      :", A2D2_ROOT)
print("metric contract: PASS")


Mounted at /content/drive
Python         : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch          : 2.8.0+cu126
GPU            : NVIDIA L4
Git            : stage3-sangchun 76391e3
Config         : /content/Blackbox-Detection/configs/stage3/vjepa21b_can_v5a.yaml
comma root     : /content/drive/MyDrive/Blackbox-Detection/DATASET/comma2k19/processed/v1
A2D2 root      : /content/drive/MyDrive/Blackbox-Detection/DATASET/A2D2/processed
metric contract: PASS


In [3]:
def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False

def copy_file_to_local(
    source: Path,
    destination: Path,
    *,
    min_bytes: int = 1,
    retries: int = 2,
) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 2):
        tmp = destination.with_name(destination.name + ".copy.tmp")
        try:
            tmp.unlink(missing_ok=True)
            with source.open("rb") as src, tmp.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
            if tmp.stat().st_size < int(min_bytes):
                raise OSError(f"staged file too small: {tmp.stat().st_size}")
            os.replace(tmp, destination)
            return True
        except OSError as exc:
            last_error = exc
            tmp.unlink(missing_ok=True)
            print(f"copy attempt {attempt} failed:", repr(exc))
            if attempt <= retries:
                time.sleep(2 * attempt)
    print("copy failed:", repr(last_error))
    return False

VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"

if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/facebookresearch/vjepa2.git",
         str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

ACTUAL_VJEPA_COMMIT = subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

CKPT_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_CKPT_DRIVE = PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT = LOCAL_PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT_URL = (
    "https://dl.fbaipublicfiles.com/vjepa2/"
    "vjepa2_1_vitb_dist_vitG_384.pt"
)
MIN_VJEPA_BYTES = 1_000_000_000

if not _is_usable_file(VJEPA_CKPT, MIN_VJEPA_BYTES):
    copied = copy_file_to_local(
        VJEPA_CKPT_DRIVE,
        VJEPA_CKPT,
        min_bytes=MIN_VJEPA_BYTES,
    )
    if not copied:
        tmp = VJEPA_CKPT.with_name(VJEPA_CKPT.name + ".download.tmp")
        tmp.unlink(missing_ok=True)
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", str(tmp), VJEPA_CKPT_URL],
            check=True,
        )
        if tmp.stat().st_size < MIN_VJEPA_BYTES:
            raise RuntimeError("V-JEPA checkpoint download is too small")
        os.replace(tmp, VJEPA_CKPT)

print("V-JEPA commit:", ACTUAL_VJEPA_COMMIT)
print("V-JEPA ckpt  :", VJEPA_CKPT)


V-JEPA commit: 45d025f636dfc58fc2426905fc4a1ab755b1c3e5
V-JEPA ckpt  : /content/pretrained/vjepa2_1_vitb_dist_vitG_384.pt


## Mixed dataset + event-balanced sampler


In [4]:
from torch.utils.data import DataLoader

from blackbox_detection.stage3.v5_dataset import (
    MixedStage3CANDataset,
    build_event_balanced_sampler,
)

dc = cfg["data"]
tc = cfg["training"]
source_cfg = dc["sources"]

a2d2_manifest = pd.read_csv(A2D2_ROOT / "manifest.csv")
print(
    "A2D2:",
    len(a2d2_manifest),
    "segments,",
    int(a2d2_manifest["num_frames"].sum()),
    "frames",
)
assert int(a2d2_manifest["num_frames"].sum()) == 9151

train_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_train.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
    {
        "name": "a2d2",
        "manifest": a2d2_manifest,
        "processed_root": A2D2_ROOT,
        **source_cfg["a2d2"],
    },
]
val_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_val_id.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
]

train_ds = MixedStage3CANDataset(
    train_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["train_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    seed=SEED,
)
val_ds = MixedStage3CANDataset(
    val_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["val_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    seed=SEED + 1,
)

sampler_cfg = dc["sampler"]
EVENT_CACHE = (
    DRIVE_ROOT
    / "manifests/stage3/v5a"
    / f"event_index_t{dc['clip_len']}_s{dc['train_window_stride']}.npz"
)

train_sampler, sampler_report = build_event_balanced_sampler(
    train_ds,
    num_samples=int(tc["max_steps_per_epoch"]) * int(tc["batch_size"]),
    event_multipliers=sampler_cfg["event_multipliers"],
    source_multipliers=sampler_cfg["source_multipliers"],
    inverse_frequency_power=sampler_cfg["inverse_frequency_power"],
    max_normalized_weight=sampler_cfg["max_normalized_weight"],
    seed=SEED,
    cache_path=EVENT_CACHE,
)

print(json.dumps(sampler_report, indent=2))

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)
val_loader = DataLoader(
    val_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

print("train windows universe :", len(train_ds))
print("val windows            :", len(val_ds))
print("sampled train / epoch  :", len(train_sampler))
print("train loader steps     :", len(train_loader))
print("val loader steps       :", len(val_loader))


A2D2: 16 segments, 9151 frames
event index cache: HIT | /content/drive/MyDrive/Blackbox-Detection/manifests/stage3/v5a/event_index_t32_s16.npz | windows=61390
{
  "dataset_fingerprint": "c8395933008baec463caa6a3b802068448d4eda2",
  "num_windows": 61390,
  "num_samples_per_epoch": 2000,
  "event_counts": {
    "hard_decel": 8568,
    "stop_start": 1564,
    "cruise": 26358,
    "hard_accel": 6743,
    "turn": 16601,
    "reversal": 1556
  },
  "source_counts": {
    "comma2k19": 60826,
    "a2d2": 564
  },
  "expected_source_share": {
    "a2d2": 0.22257247658659413,
    "comma2k19": 0.7774275234134058
  },
  "expected_event_share": {
    "cruise": 0.09239051904008222,
    "hard_accel": 0.18725725918574593,
    "hard_decel": 0.2821178674604734,
    "reversal": 0.07434031420451512,
    "stop_start": 0.12908704568110776,
    "turn": 0.23480699442807557
  },
  "weight_min": 0.19171301353919537,
  "weight_mean": 0.9429399933021453,
  "weight_max": 30.0
}
train windows universe : 61390
val w

## Model — v4 head warm-start + spatial residual + T=32


In [5]:
from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.v5_models import (
    VJEPA21DenseCANV5,
    load_v4_head_warm_start,
)
from blackbox_detection.stage3.trainer import build_scheduler
from blackbox_detection.stage3.v5_trainer import V5CANTrainer

mc = cfg["model"]
fusion_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_CKPT,
    num_frames=dc["clip_len"],
    out_layers=tuple(mc["out_layers"]),
    freeze=mc["freeze_backbone"],
)

model = VJEPA21DenseCANV5(
    backbone,
    freeze_backbone=mc["freeze_backbone"],
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=spatial_cfg["gate_init"],
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=fusion_cfg["enabled"],
    accel_fusion_hidden=fusion_cfg["hidden"],
    accel_fusion_gate_init=fusion_cfg["gate_init"],
    accel_fusion_detach_ordinal_inputs=fusion_cfg["detach_ordinal_inputs"],
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Stage this run's own checkpoints first.
for filename in ("latest.pt", "best.pt"):
    persistent = RUN_DIR / filename
    local = LOCAL_RUN_DIR / filename
    if not local.is_file() and _is_usable_file(persistent):
        print(
            "stage own", filename,
            copy_file_to_local(persistent, local),
        )

resume_path = LOCAL_RUN_DIR / "latest.pt"
warm_start_report = None

if not resume_path.is_file():
    wc = cfg["warm_start"]
    v4_drive = OUTPUT_ROOT / wc["run_name"] / wc["checkpoint"]
    v4_local = (
        LOCAL_PRETRAINED_ROOT
        / f"{wc['run_name']}__{wc['checkpoint']}"
    )
    if not _is_usable_file(v4_local, 1_000_000):
        if not _is_usable_file(v4_drive, 1_000_000):
            raise FileNotFoundError(v4_drive)
        if not copy_file_to_local(
            v4_drive, v4_local, min_bytes=1_000_000
        ):
            raise OSError("failed to stage v4 best checkpoint")

    warm_start_report = load_v4_head_warm_start(model, v4_local)
    print("V4 -> V5 head warm-start:")
    print(json.dumps(
        {
            k: v
            for k, v in warm_start_report.items()
            if k != "copied_head_keys"
        },
        indent=2,
        default=str,
    ))
else:
    print("V5 latest.pt found: full V5 resume will take precedence")

new_tokens = (
    "spatial_pool",
    "stop_ordinal_head",
    "delta_speed_head",
    "turn_ordinal_head",
    "steer_direction_head",
    "steer_activity_ordinal_head",
    "brake_ordinal_head",
    "throttle_ordinal_head",
)

base_lr = float(tc["base_learning_rate"])
new_lr = float(tc["new_learning_rate"])
weight_decay = float(tc["weight_decay"])

buckets = {
    "base_decay": [],
    "base_no_decay": [],
    "new_decay": [],
    "new_no_decay": [],
}
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    is_new = any(token in name for token in new_tokens)
    no_decay = p.ndim <= 1 or name.endswith(".bias")
    family = "new" if is_new else "base"
    decay = "no_decay" if no_decay else "decay"
    buckets[f"{family}_{decay}"].append(p)

optimizer_groups = []
for group_name, params in buckets.items():
    if not params:
        continue
    is_new = group_name.startswith("new")
    no_decay = group_name.endswith("no_decay")
    optimizer_groups.append(
        {
            "params": params,
            "lr": new_lr if is_new else base_lr,
            "weight_decay": 0.0 if no_decay else weight_decay,
            "name": group_name,
        }
    )

optimizer = torch.optim.AdamW(optimizer_groups)

micro_steps_per_epoch = min(
    len(train_loader),
    int(tc["max_steps_per_epoch"]),
)
optimizer_steps_per_epoch = max(
    math.ceil(
        micro_steps_per_epoch / int(tc["grad_accum_steps"])
    ),
    1,
)
total_optimizer_steps = (
    optimizer_steps_per_epoch * int(tc["epochs"])
)

scheduler = build_scheduler(
    optimizer,
    total_steps=total_optimizer_steps,
    warmup_ratio=tc["warmup_ratio"],
    min_ratio=tc["min_learning_rate_ratio"],
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
new_params = sum(
    p.numel()
    for name, p in model.named_parameters()
    if p.requires_grad and any(token in name for token in new_tokens)
)
print("trainable params:", trainable_params / 1e6, "M")
print("new v5 params    :", new_params / 1e6, "M")
print("base lr / new lr:", base_lr, "/", new_lr)
for g in optimizer.param_groups:
    print(g["name"], "lr=", g["lr"], "wd=", g["weight_decay"])


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


stage own latest.pt True
stage own best.pt True
V5 latest.pt found: full V5 resume will take precedence
trainable params: 4.844968 M
new v5 params    : 1.783066 M
base lr / new lr: 3e-05 / 0.0002
base_decay lr= 7.894736842105263e-07 wd= 0.01
base_no_decay lr= 7.894736842105263e-07 wd= 0.0
new_decay lr= 5.263157894736842e-06 wd= 0.01
new_no_decay lr= 5.263157894736842e-06 wd= 0.0


## W&B + trainer


In [6]:
lc = cfg.get("logging", {})
WANDB_ENABLED = bool(lc.get("wandb_enabled", True))
wandb_run = None

if WANDB_ENABLED:
    import wandb
    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(WANDB_KEY_PATH)
    wandb.login(
        key=WANDB_KEY_PATH.read_text(encoding="utf-8").strip(),
        relogin=False,
    )
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = (
        run_id_path.read_text(encoding="utf-8").strip()
        if run_id_path.is_file()
        else None
    )
    stored_run_id = stored_run_id or None

    wandb_run = init_wandb(
        project=str(lc.get("wandb_project", "blackbox-stage3")),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}",
        group=str(lc.get("wandb_group", "vjepa21b_can_v5a")),
        tags=[
            "stage3", "vjepa2.1", "v5a", "a2d2",
            "spatial-moments", "t32", "event-balanced",
            "frozen-backbone",
        ],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "vjepa_commit": ACTUAL_VJEPA_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "sampler_report": sampler_report,
        },
    )
    if not stored_run_id and wandb_run is not None:
        run_id_path.write_text(str(wandb_run.id), encoding="utf-8")

vc = cfg.get("validation", {})
trainer = V5CANTrainer(
    model,
    optimizer,
    scheduler=scheduler,
    device="cuda" if torch.cuda.is_available() else "cpu",
    grad_accum_steps=tc["grad_accum_steps"],
    grad_clip_norm=tc["grad_clip_norm"],
    amp_dtype=tc["amp_dtype"],
    loss_weights=loss_cfg,
    stats=stats,
    proxy_rules=vc.get("proxy_rules", {}),
    output_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    wandb_enabled=WANDB_ENABLED,
    log_interval=tc["log_interval"],
    config=cfg,
)
print("trainer device:", trainer.device)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sangchun1 (sangchun1-chung-ang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


trainer device: cuda


## One-batch smoke — 반드시 PASS 후 epoch 시작


In [ ]:
from blackbox_detection.stage3.v5_losses import v5_multitask_loss

smoke = next(iter(train_loader))
video = smoke["video"][:1].to(trainer.device)
target = smoke["target"][:1].to(trainer.device)
valid = smoke["valid"][:1].to(trainer.device)
aux = {
    k: v[:1].to(trainer.device)
    for k, v in smoke["aux"].items()
}

optimizer.zero_grad(set_to_none=True)
model.train()

with torch.autocast(
    device_type=trainer.device.type,
    dtype=trainer.amp_dtype,
    enabled=trainer.device.type == "cuda",
):
    out = model(video)
    smoke_loss, smoke_parts = v5_multitask_loss(
        out,
        target,
        valid,
        aux,
        loss_cfg,
        stats=stats,
    )

assert torch.isfinite(smoke_loss), smoke_loss
assert tuple(out["stop_ordinal_logits"].shape) == (
    1, dc["clip_len"], len(mc["stop_thresholds_mps"])
)
assert tuple(out["turn_ordinal_logits"].shape) == (
    1, dc["clip_len"], len(mc["turn_yaw_thresholds_rps"]), 2
)
assert tuple(out["steer_direction_logits"].shape) == (
    1, dc["clip_len"], 3
)

smoke_loss.backward()

spatial_grad = model.spatial_pool.spatial_proj.weight.grad
assert spatial_grad is not None
assert torch.isfinite(spatial_grad).all()
assert float(spatial_grad.float().norm().cpu()) > 0

stop_grad = model.head.stop_ordinal_head.weight.grad
assert stop_grad is not None
assert torch.isfinite(stop_grad).all()

print("SMOKE PASS")
print("loss                :", float(smoke_loss.detach().cpu()))
print("spatial grad norm   :", float(spatial_grad.float().norm().cpu()))
print("stop grad norm      :", float(stop_grad.float().norm().cpu()))
print("spatial gate        :", float(
    torch.sigmoid(model.spatial_pool.gate_logit).detach().cpu()
))
print(json.dumps(smoke_parts, indent=2))

optimizer.zero_grad(set_to_none=True)
del video, target, valid, aux, out, smoke_loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Train / resume


In [7]:
from torch.utils.data import DataLoader, Subset

# 전체 validation windows에서 deterministic하게 골고루 800개 선택
VAL_WINDOWS = min(
    int(tc["max_val_steps"]),
    len(val_ds),
)

val_indices = np.linspace(
    0,
    len(val_ds) - 1,
    num=VAL_WINDOWS,
    dtype=np.int64,
)

# 혹시 정수화 때문에 중복이 생기지 않았는지 확인
val_indices = np.unique(val_indices)

val_eval_ds = Subset(
    val_ds,
    val_indices.tolist(),
)

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)

val_loader = DataLoader(
    val_eval_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

print("train windows universe :", len(train_ds))
print("val windows universe   :", len(val_ds))
print("val windows evaluated  :", len(val_eval_ds))
print("sampled train / epoch  :", len(train_sampler))
print("train loader steps     :", len(train_loader))
print("val loader steps       :", len(val_loader))

train windows universe : 61390
val windows universe   : 12841
val windows evaluated  : 800
sampled train / epoch  : 2000
train loader steps     : 2000
val loader steps       : 800


In [9]:
resume_path = LOCAL_RUN_DIR / "latest.pt"
print("resume:", resume_path if resume_path.is_file() else None)

history = trainer.fit(
    train_loader,
    val_loader,
    epochs=tc["epochs"],
    max_train_steps=tc["max_steps_per_epoch"],
    max_val_steps=tc["max_val_steps"],
    resume_from=resume_path if resume_path.is_file() else None,
    early_stopping_patience=tc.get("early_stopping_patience", 0),
    backfill_validation_on_resume=False,
)

history_df = pd.DataFrame([
    {
        "epoch": row["epoch"],
        "minutes": row["minutes"],
        "learning_rate": row.get("learning_rate"),
        "max_gpu_memory_gib": row.get("max_gpu_memory_gib"),
        **{f"train/{k}": v for k, v in row["train"].items()},
        **{f"val/{k}": v for k, v in row["val"].items()},
    }
    for row in history
])
display(history_df)


resume: /content/stage3_runs/vjepa21b_can_v5a_spatial_t32_a2d2/latest.pt
[2026-09-24 17:04:19] INFO | blackbox_detection.stage3.can | Resumed from /content/stage3_runs/vjepa21b_can_v5a_spatial_t32_a2d2/latest.pt at epoch 2 (next=3, best val total=1.739029, global_step=500).


Train 3:   0%|          | 0/2000 [00:00<?, ?it/s]

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Val 3:   0%|          | 0/800 [00:00<?, ?it/s]

[2026-09-24 17:33:40] INFO | blackbox_detection.stage3.can | Epoch 3/3 | train 2.301146 | val 1.681318 | proxy-F1(mean) 0.6381 | lr 1.500e-06 | 29.3 min | peak 0.72 GiB <- best CAN-pretrain


,epoch,minutes,learning_rate,max_gpu_memory_gib,train/base/accel_from_speed_mps2,train/base/accel_v2/mean_sample_weight,train/base/accel_v2/multi_threshold_margin,train/base/accel_v2/speed_accel_consistency,train/base/accel_v2/speed_delta,train/base/accel_v2/weighted_regression,...,val/proxy/conservative/f1_accel_CONSTANT,val/proxy/conservative/f1_accel_STOPPED,val/proxy/conservative/f1_steer_LEFT,val/proxy/conservative/f1_steer_STRAIGHT,val/proxy/conservative/f1_steer_RIGHT,val/proxy/robust_mean_stage3_score,val/proxy/robust_min_stage3_score,val/proxy/robust_max_stage3_score,val/proxy/robust_mean_accel_macro_f1,val/proxy/robust_mean_steer_macro_f1
0,1,31.861615,0.000024,0.719127,0.538062,2.192651,0.325230,0.204965,0.005139,0.565458,...,0.848280,0.934283,0.493252,0.970986,0.347233,0.607398,0.584639,0.640254,0.609614,0.602227
1,2,12.760516,0.000009,0.719615,0.497196,2.183001,0.283097,0.203253,0.005349,0.523162,...,0.844863,0.930188,0.511828,0.967668,0.350305,0.623866,0.610841,0.639046,0.624171,0.623155
2,3,29.259728,0.000002,0.720935,0.448982,2.192651,0.271718,0.201093,0.005368,0.472673,...,0.851329,0.922262,0.557752,0.967567,0.334007,0.638065,0.626732,0.654740,0.640927,0.631388


## v4-A vs v5-A diagnostics


In [10]:
if len(history_df):
    proxy_col = "val/proxy/robust_mean_stage3_score"
    accel_col = "val/proxy/robust_mean_accel_macro_f1"

    best_proxy_idx = (
        pd.to_numeric(history_df[proxy_col], errors="coerce").idxmax()
        if proxy_col in history_df
        else history_df["val/total"].astype(float).idxmin()
    )
    best_row = history_df.loc[best_proxy_idx]

    summary = {
        "run_variant": RUN_VARIANT,
        "git_commit": GIT_COMMIT,
        "vjepa_commit": ACTUAL_VJEPA_COMMIT,
        "clip_len": dc["clip_len"],
        "sampler_report": sampler_report,
        "diagnostic_best_epoch": int(best_row["epoch"]),
        "diagnostic_best_proxy_stage3": (
            float(best_row[proxy_col])
            if proxy_col in history_df
            else None
        ),
        "diagnostic_best_proxy_accel": (
            float(best_row[accel_col])
            if accel_col in history_df
            else None
        ),
        "note": (
            "proxy thresholds are diagnostic only; official DACON metric "
            "labels remain unavailable offline"
        ),
    }

    v4_summary_path = (
        OUTPUT_ROOT
        / cfg["warm_start"]["run_name"]
        / "summary.json"
    )
    if v4_summary_path.is_file():
        v4_summary = json.loads(
            v4_summary_path.read_text(encoding="utf-8")
        )
        for key in [
            "best_val/diag/accel/pred_to_gt_std_ratio",
            "best_val/diag/accel/correlation",
            "best_val/proxy/robust_mean_accel_macro_f1",
            "best_val/proxy/robust_mean_stage3_score",
        ]:
            if key in v4_summary:
                summary[f"v4a_{key}"] = v4_summary[key]

    local_summary = LOCAL_RUN_DIR / "summary.json"
    local_summary.write_text(
        json.dumps(summary, indent=2, default=str),
        encoding="utf-8",
    )
    trainer._sync_file(local_summary)
    print(json.dumps(summary, indent=2))

if WANDB_ENABLED:
    finish_wandb()

print("persistent latest:", RUN_DIR / "latest.pt")
print("persistent best  :", RUN_DIR / "best.pt")


{
  "run_variant": "vjepa21b_can_v5a_spatial_t32_a2d2",
  "git_commit": "76391e3",
  "vjepa_commit": "45d025f636dfc58fc2426905fc4a1ab755b1c3e5",
  "clip_len": 32,
  "sampler_report": {
    "dataset_fingerprint": "c8395933008baec463caa6a3b802068448d4eda2",
    "num_windows": 61390,
    "num_samples_per_epoch": 2000,
    "event_counts": {
      "hard_decel": 8568,
      "stop_start": 1564,
      "cruise": 26358,
      "hard_accel": 6743,
      "turn": 16601,
      "reversal": 1556
    },
    "source_counts": {
      "comma2k19": 60826,
      "a2d2": 564
    },
    "expected_source_share": {
      "a2d2": 0.22257247658659413,
      "comma2k19": 0.7774275234134058
    },
    "expected_event_share": {
      "cruise": 0.09239051904008222,
      "hard_accel": 0.18725725918574593,
      "hard_decel": 0.2821178674604734,
      "reversal": 0.07434031420451512,
      "stop_start": 0.12908704568110776,
      "turn": 0.23480699442807557
    },
    "weight_min": 0.19171301353919537,
    "weight_mean

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
optim/learning_rate,██▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
selection/best_val_total,▁
system/epoch_minutes,▁
system/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
system/max_gpu_memory_gib,▁
train/base/accel_from_speed_mps2,▁
train/base/accel_v2/mean_sample_weight,▁
train/base/accel_v2/multi_threshold_margin,▁
train/base/accel_v2/speed_accel_consistency,▁
+219,...


persistent latest: /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5a_spatial_t32_a2d2/latest.pt
persistent best  : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5a_spatial_t32_a2d2/best.pt
